<a href="https://colab.research.google.com/github/haowu0916edisonwu/haowu0916edisonwu.github.io/blob/main/Caredemo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from google.colab import drive

print("开始初始化 L4 环境...")

# 1. 挂载云盘 (防止断开连接后数据丢失)
drive.mount('/content/drive')

# 2. 创建并进入工作目录
project_root = '/content/drive/MyDrive/CARE_L4_Full'
os.makedirs(project_root, exist_ok=True)
%cd {project_root}

# 3. 克隆代码
if not os.path.exists("CARE"):
    !git clone https://github.com/eunseongc/CARE.git
%cd CARE
%ls

开始初始化 L4 环境...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/CARE_L4_Full
/content/drive/MyDrive/CARE_L4_Full/CARE
config/                evaluate_closedbook.sh  ft_results/  requirements.txt
data_preprocess.ipynb  evaluate.sh             pretrain.sh  src/
eun_temp.ipynb         finetune.sh             README.md


In [2]:
print("🛑 正在执行【强制跳过编译】方案...")

# 1. 先把所有文件还原到初始状态 (清除之前的修改)
!git checkout .

# 2. 【核心操作】直接从 requirements.txt 中删掉 flash-attn
# 这样 pip install 时就会完全无视它，瞬间完成！
!sed -i '/flash-attn/d' requirements.txt

print("📦 3. 安装依赖 (极速版 - 无需编译)...")
# 这次安装会飞快，因为只装普通包
!pip install -r requirements.txt
!pip install bitsandbytes

🛑 正在执行【强制跳过编译】方案...
Updated 23 paths from the index
📦 3. 安装依赖 (极速版 - 无需编译)...


In [3]:
from huggingface_hub import login
import os

# 1. 登录 (如果之前没登录过)
print("请在下方输入 Hugging Face Token:")
login()

请在下方输入 Hugging Face Token:


In [4]:
print("下载模型和数据...")
# 下载权重
!huggingface-cli download eunseong/care_mistral_pt --local-dir checkpoints/care_mistral_pt
# 下载数据
!huggingface-cli download eunseong/data_care --repo-type dataset --local-dir data_care --local-dir-use-symlinks False

# 准备 Demo 文件
!mkdir -p data_care/finetune
!cp -f data_care/finetune/nq_mistral.jsonl data_care/finetune/nq_mistral.jsonl_demo
!cp -f data_care/finetune/nq_valid.jsonl data_care/finetune/nq_valid.jsonl_demo

print("数据准备完毕")

下载模型和数据...
⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]Downloading 'ckpt.pth' to 'checkpoints/care_mistral_pt/.cache/huggingface/download/I5mKl6660ZnMyBqW_iS6OPv3c_M=.0522ef56a7571452ebb09574c01eb85281b69d0d0ba4374503a7a9230c1b25d5.incomplete'

ckpt.pth:   0% 0.00/1.01G [00:00<?, ?B/s]

config.json: 2.97kB [00:00, 4.42MB/s]
Download complete. Moving file to checkpoints/care_mistral_pt/config.json


README.md: 100% 31.0/31.0 [00:00<00:00, 174kB/s]
Download complete. Moving file to checkpoints/care_mistral_pt/README.md


.gitattributes: 1.52kB [00:00, 6.44MB/s]
Download complete. Moving file to checkpoints/care_mistral_pt/.gitattributes
Fetching 4 files:  25% 1/4 [00:00<00:02,  1.18it/s]
ckpt.pth:   0% 1.06M/1.01G [00:03<55:03, 305kB/s]
ckpt.pth:   7% 68.1M/1.01G [00:07<01:26, 10.9MB/s]
ckpt.pth:  20% 202M/1.01G [00:07<00:20, 40.0MB/s] 
ckpt.pth:  33% 336M/1.01G [00:07<00:08, 77.7MB/s]
ckpt.pth:  47% 470M/1.

In [5]:
print("配置 L4 代码 (BF16 + 无FlashAttn)...")

# 1. 准备 Demo 数据
!mkdir -p data_care/finetune
!cp -f data_care/finetune/nq_mistral.jsonl data_care/finetune/nq_mistral.jsonl_demo
!cp -f data_care/finetune/nq_valid.jsonl data_care/finetune/nq_valid.jsonl_demo

# 2. [修复] args.py 原生 Bug
!sed -i 's/if args.eval_file is not None:/if hasattr(args, "eval_file") and args.eval_file is not None:/g' src/utils/args.py

# 3. [关键] 强制移除代码中的 Flash Attention 调用
# 原代码里写死了 use_flash_attention_2=True，删掉它让 transformers 自动选择
!sed -i '/use_flash_attention_2=/d' src/model/ICAE/modeling_icae.py

# 4. [关键] 注入 4-bit 量化
# 配合 BF16 使用，既稳又快，且极省显存
!sed -i 's/config=config)/config=config, load_in_4bit=True)/g' src/model/ICAE/modeling_icae.py

# 5. [关键] 配置文件禁用 Flash Attention
!sed -i 's/use_flash_attn: true/use_flash_attn: false/g' config/language_modeling/finetune.yaml

print("配置完成：已彻底剥离 Flash Attention，启用 BF16 + 4-bit。")

配置 L4 代码 (BF16 + 无FlashAttn)...
配置完成：已彻底剥离 Flash Attention，启用 BF16 + 4-bit。


In [12]:
print("🧹 正在修复缩进错误...")

# 1. 仅重置 train.py (保留其他文件的修改)
!git checkout src/language_modeling/train.py

# 2. [补丁1] 再次应用 strict=False (加载权重必须)
!sed -i 's/model.load_state_dict(model_state_dict)/model.load_state_dict(model_state_dict, strict=False)/g' src/language_modeling/train.py

# 3. [补丁2] 完美替换验证代码 (保持缩进)
# 使用正则表达式替换整行调用，将其变为 score = 0.0
# 因为没有匹配行首的空格，sed 会保留原有的缩进，这解决了 IndentationError
!sed -i 's/score = validate_during_finetune(.*)/score = 0.0/g' src/language_modeling/train.py

print("✅ train.py 已修复：Validation 被安全跳过，缩进正常。")
print("----------------------------------------------------------------")
print("🚀 10. 启动 Demo (见证奇迹时刻)...")

# 启动命令
!export WANDB_MODE=offline && accelerate launch \
    --mixed_precision bf16 \
    --num_processes 1 \
    -m src.language_modeling.train \
    --config config/language_modeling/finetune.yaml \
    --dev_file data_care/finetune/nq_valid.jsonl \
    --train_file data_care/finetune/nq_mistral.jsonl \
    --model_name_or_path mistralai/mistral-7b-instruct-v0.2 \
    --checkpoint_path checkpoints/care_mistral_pt \
    --learning_rate 1e-4 \
    --num_train_epochs 1 \
    --max_train_steps 20 \
    --per_device_train_batch_size 1 \
    --gradient_accumulation_steps 32 \
    --alpha_kl 2 \
    --exp_name care_mistral_demo_L4 \
    --demo True

🧹 正在修复缩进错误...
Updated 1 path from the index
✅ train.py 已修复：Validation 被安全跳过，缩进正常。
----------------------------------------------------------------
🚀 10. 启动 Demo (见证奇迹时刻)...
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specifi